In [ ]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

def remove_zeros(array):
    a = onp.array(array)
    a[onp.abs(a) < 1e-14] = 0.0
    return a

from jax_sn.domain.geometric_domain import GeometricDomain, TopologicalSortedAngle
from jax_sn.solution_domain.solution_domain import FEMMatrices, PrecomputedGeometricData
from jax_sn.domain.geometric_domain import CrossSectionData, CrossSectionDataSingleGroup
from jax_sn.source import Source
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)

In [ ]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain

In [ ]:
from jax_sn.solution_domain import compute_setup_for_solution_domain, transport_inverse_sweep_complete_vmap_fori
from jax_sn.solution_domain.transport_operator import transport_inverse_sweep_multiple_angles_batched, transport_inverse_sweep_multiple_angles_batched_single_f


sigma  = 5
xs_0   = CrossSectionData.constant(sigma, 1, 1)
q      = 5
dim    = 3
degree = 3
n_elem = 100
verts, conn, cross_sections, sources = create_cartesian_setup(dim, [[(1.0, n_elem)], [(0.2, 10)], [(0.2, 10)]], cross_sections_regions=[xs_0], source_regions=[q], inhomogeneity_direction=0)
element_type                         = BasixLagrangianSimplex(degree, dim)
domain                               = Domain.from_mesh_and_cross_sections((verts, conn), element_type.face_template, cross_sections_indexed=cross_sections)
solution_domain                      = SolutionDomain.from_element_and_domain(element_type, domain)
source_mapped                        = Source(jnp.repeat(sources, element_type.n_basis, axis=-1).reshape(-1, element_type.n_basis))
buffer_sweep                         = jnp.zeros((domain.n_elements, element_type.n_basis))

quad_set = jsn.quadrature_set.create_tn_quadrature_set(4)
toposorts = [TopologicalSortedAngle.from_domain_angle(domain, angle, 100) for angle in quad_set.angles]
total_toposorts = jax.tree.map(lambda * xs : jnp.stack(xs), *toposorts)

In [ ]:
source_angle = jax.tree.map(lambda ts : jnp.repeat(ts, total_toposorts.angle.shape[0], axis=0).reshape((total_toposorts.angle.shape[0],)+ts.shape), source_mapped)

In [ ]:
total_buffer_sweep = jnp.zeros((total_toposorts.angle.shape[0], domain.n_elements, element_type.n_basis))

In [ ]:
from jax_sn.solution_domain.transport_operator import _setup_elements_idx_in_precomp_with_omega_n_calculation
@jax.jit
def transport_inverse_sweep_complete_vmap_fori(geometric_domain           : GeometricDomain, 
                                               fem_matrices               : FEMMatrices, 
                                               precomputed_geometric_data : PrecomputedGeometricData, 
                                               cross_sections             : CrossSectionDataSingleGroup,                                                
                                               topological_sorted_angle   : TopologicalSortedAngle, 
                                               precomputed_source         : Source, 
                                               sweep_result               : jnp.ndarray) -> jnp.ndarray:    
    '''
    Perform a complete transport inverse sweep over all elements in the geometric domain using vmap and fori_loop.

    vmap is used to compute all Gij_inv, QM_i and Dij_f_p_fm matrices for all elements in a vectorized manner.
    Then, a fori_loop is used to iterate over the elements in topological order to compute the sweep result.
    This takes significant memory due to the large intermediate matrices. 

    Parameters
    ----------
    geometric_domain : GeometricDomain
        The geometric domain data.
    fem_matrices : FEMMatrices
        The finite element method matrices.
    precomputed_geometric_data : PrecomputedGeometricData
        The precomputed geometric data (Jacobian determinants and inverses).
    cross_sections : CrossSectionDataSingleGroup
        The cross section data.
    topological_sorted_angle : TopologicalSortedAngle
        The topologically sorted angle data.
    precomputed_source : Source
        The source data.
    sweep_result : jnp.ndarray [n_elements, n_basis]
        The initial sweep result array (usually zeros).
    Returns
    -------
    sweep_result : jnp.ndarray [n_elements, n_basis]    
        The computed sweep result array after performing the transport sweep.
    '''    
    

    gij_inv_all, qm_i_all, dij_f_p_fm_all, _, abs_omega_dot_n_Je = _setup_elements_idx_in_precomp_with_omega_n_calculation(topological_sorted_angle.levels_argsort, geometric_domain, fem_matrices,
                                                            precomputed_geometric_data, cross_sections, topological_sorted_angle.angle,
                                                            precomputed_source)    
    return gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je



In [ ]:

gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je = transport_inverse_sweep_complete_vmap_fori(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), toposorts[0], source_mapped, buffer_sweep)

In [ ]:



def iota_kernel(topo_sort, o_ref):

  def _body(i, o_ref):
      o_ref[i] = topo_sort[i]
      return o_ref

  jax.lax.fori_loop(0, topo_sort.shape[0], _body, o_ref)


@partial(jax.jit, static_argnums=(0,))
def iota(size: int):
  return pl.pallas_call(iota_kernel,
                        out_shape=jax.ShapeDtypeStruct((size,), jnp.int32),
                        grid=(1,))(toposorts[0].levels_argsort)



In [ ]:
solution_domain.domain.n_elements

In [ ]:
%timeit iota(solution_domain.domain.n_elements).block_until_ready()

In [ ]:
iota(solution_domain.domain.n_elements).shape

In [ ]:
toposorts[0]

In [ ]:
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl
def sweep_pallas(elements_topological_sorting, gij_inv_all, qm_i_all,
                  dij_f_p_fm_all, abs_omega_dot_n_Je, etoe,
                  sweep_result_init):

    # capture all data in a closure
    @jax.jit
    def kernel(o_ref):
        i = 0
        N = elements_topological_sorting.shape[0]

        
        elem_idx = elements_topological_sorting[i]

        # gij_inv    = gij_inv_all[i]
        # qm_i       = qm_i_all[i]
        # dij_matrix = dij_f_p_fm_all[i]
        # omega      = abs_omega_dot_n_Je[i]
        # neighbors  = etoe[elem_idx]

        # values = o_ref[neighbors]

        # # collapsed_dij = jnp.einsum("f, f i j, f j -> i",
        # #                             omega, dij_matrix, values)

        # result = gij_inv @ (qm_i )#+ collapsed_dij)

        o_ref[elem_idx] = result

        #return o_ref
    def ree():
        return pl.pallas_call(
            kernel,
            out_shape=jax.ShapeDtypeStruct(sweep_result_init.shape,
                                       sweep_result_init.dtype),
            grid=(1,),  # scalar grid
        )
    
    return ree()


result = sweep_pallas(
    elements_topological_sorting=toposorts[0].levels_argsort,
    gij_inv_all=gij_inv_all,
    qm_i_all=qm_i_all,
    dij_f_p_fm_all=dij_f_p_fm_all,
    abs_omega_dot_n_Je=abs_omega_dot_n_Je,
    etoe=solution_domain.domain.etoe,
    sweep_result_init=buffer_sweep
)

In [ ]:
result()

In [ ]:
%timeit transport_inverse_sweep_complete_vmap_fori(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), toposorts[0], source_mapped, buffer_sweep)[3].block_until_ready()

In [ ]:
from jax.experimental import pallas as pl



def 

In [ ]:
result_bs = transport_inverse_sweep_multiple_angles_batched_single_f(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), total_toposorts, source_angle, total_buffer_sweep, 70).block_until_ready()

In [ ]:
result_bs = transport_inverse_sweep_multiple_angles_batched(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), total_toposorts, source_angle, total_buffer_sweep, 100).block_until_ready()

In [ ]:
topolevels = [jsn.solution_domain.transport_operator_levels.TopologicalSortedAnglePrecomputedLevels.from_topological_sorted_angle(ts) for ts in toposorts]

In [ ]:
sizes = [tl.sizes for tl in topolevels]

In [ ]:
print(sizes)

In [ ]:
jnp.min(jnp.bincount(topolevels[0].levels))

In [ ]:
for i in topolevels:
    print(jnp.min(i.sizes))

In [ ]:
n_max = onp.max([onp.max(tl.level_log2_size) for tl in topolevels])
print(n_max)

In [ ]:
buffer_sweep_dummy = jnp.zeros((domain.n_elements + 1, element_type.n_basis))


In [ ]:
for i in range(len(topolevels)):
    %timeit jsn.solution_domain.transport_operator_levels.transport_inverse_sweep_complete_vmap_levels_max(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevels[i], source_mapped, buffer_sweep_dummy, n_max).block_until_ready()

In [ ]:
print([topolevels[i].sizes for i in range(len(topolevels))])

In [ ]:
[jsn.solution_domain.transport_operator_levels.transport_inverse_sweep_complete_vmap_levels_max(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevels[i], source_mapped, buffer_sweep_dummy, n_max).block_until_ready() for i in range(len(topolevels))]

In [ ]:
jsn.solution_domain.transport_operator_levels.transport_inverse_sweep_complete_vmap_levels_max(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevels[0], source_mapped, buffer_sweep_dummy, n_max).block_until_ready()

In [ ]:

from jax_sn.solution_domain.transport_operator import _collapse_dij, _collapse_dij_vmap, _assert_shape_base

def _single_level_from_setup(geometric_domain : GeometricDomain,

In [ ]:
%timeit transport_inverse_sweep_complete_vmap_levels_max(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevel, source_mapped, buffer_sweep_dummy, int(onp.max(topolevel.level_log2_size))).block_until_ready()

In [ ]:
%timeit _setup_elements_idx_in_precomp_with_omega_n_calculation(topolevel.levels_argsort, *solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevel.angle, source_mapped)[0].block_until_ready()

In [ ]:
%timeit  transport_inverse_sweep_complete_vmap_levels_with_callables(*solution_domain.solution_domain_data, solution_domain.map_cross_sections_to_full_array(0), topolevel, source_mapped, buffer_sweep_dummy).block_until_ready()

In [ ]:
collapse_new = collapse(solution_domain.domain, toposorts[0], buffer_sweep, gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je).block_until_ready()

In [ ]:
%timeit collapse(solution_domain.domain, toposorts[0], buffer_sweep, gij_inv_all, qm_i_all, dij_f_p_fm_all, abs_omega_dot_n_Je).block_until_ready()

In [ ]:
onp.testing.assert_allclose(collapse_new, buffer_sweep_dummy[:-1])

In [ ]:
from jax_sn.solution_domain.transport_operator import _sweep_elements
%timeit _sweep_elements(toposorts[0].levels_argsort, buffer_sweep, gij_inv_all, qm_i_all, dij_f_p_fm_all, solution_domain.domain, abs_omega_dot_n_Je).block_until_ready()

In [ ]:
from jax_sn.solution_domain.transport_operator import transport_inverse_sweep_multiple_angles_batched, transport_inverse_sweep_multiple_angles_batched_single_f

In [ ]:
from stell_blanket_functions import  _matfree_nte_from_sb, _test_domain_from_sb

In [ ]:
def check_sb():
    domain = _test_domain_from_sb(3,2)
    import StellBlanket as SB
    quad_set = SB.Generic_Elements.TN_Quadrature_Set(4)
    _, source = _matfree_nte_from_sb(3, 2, 4, 0)
    simulation = SB.Generic_Elements.Create_RAM_Simulation(quad_set, SB.Generic_Elements.Energy_Set([0,1]), domain, 2, 0)
    print(simulation.geometric_domain.connectivity.shape[0])
    simulation.simulation.Sweep(simulation.energy_set[0], simulation.quadrature_set[0], source)
    %timeit simulation.simulation.Total_Sweep(simulation.energy_set[0], source)

check_sb()